<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Segmentation — Implementation</b></h1>
</div>

## Setup — Environment and Configuration

In [ ]:
# Import the numerical and image-processing tools used across the segmentation pipeline.
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from scipy import ndimage

import cv2

np.set_printoptions(precision=3, suppress=True)

print("NumPy :", np.__version__)
print("OpenCV:", cv2.__version__)
print("Setup : PASS")

### 0.1 Locate the Lab Automatically


In [ ]:
def find_lab_root():
    """Find the segmentation lab without depending on Jupyter's start folder."""
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "notebooks" / "main.ipynb").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the segmentation lab root."
    )


LAB_ROOT = find_lab_root()
DATA_DIR = LAB_ROOT / "data"
OUTPUT_DIR = LAB_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Discover all supported images recursively; dataset size is not hard-coded.
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
DATA_FILES = sorted(
    path for path in DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

assert DATA_FILES, f"No supported images found in: {DATA_DIR}"

# Resolve a semantic experiment role from the discovered dataset, not from a path list.
def select_image(role: str) -> Path:
    role = role.lower()

    exact = [
        path for path in DATA_FILES
        if path.stem.lower() == role
    ]
    if exact:
        return exact[0]

    matches = [
        path for path in DATA_FILES
        if role in path.stem.lower()
    ]
    if not matches:
        raise FileNotFoundError(
            f"No image matching role '{role}' found in {DATA_DIR}"
        )
    return matches[0]


INPUT_IMAGES = {
    role: select_image(role)
    for role in ("hand", "tower", "peppers")
}

print("Lab root         :", LAB_ROOT)
print("Data dir         :", DATA_DIR)
print("Images discovered:", len(DATA_FILES))
print("Experiment roles :", len(INPUT_IMAGES))
print("Output           :", OUTPUT_DIR)

### 0.2 Reusable helpers

In [ ]:
# Centralize image I/O so segmentation operates on consistent dtype/channel conventions.
def load_gray(path):
    """Load one image when segmentation will be driven by intensity.
    
    Grayscale removes color as a variable and leaves one scalar field for
    thresholding, morphology, distance transforms, and connected components."""
    return np.asarray(
        Image.open(path).convert("L"),
        dtype=np.uint8
    )


def load_rgb(path):
    """Load one image when color itself is part of the segmentation evidence.
    
    We keep RGB only for experiments where chromatic information or overlays matter."""
    return np.asarray(
        Image.open(path).convert("RGB"),
        dtype=np.uint8
    )


def show_gray(ax, image, title):
    """Display a scalar image or binary mask consistently.
    
    Using one helper keeps the visual language stable across thresholds, masks,
    distance maps, and morphology results."""
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")


def show_rgb(ax, image, title):
    """Display an RGB image without repeated plotting setup.
    
    The helper keeps image presentation consistent while the actual experiment
    remains visible in the data, not in styling differences."""
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")


def save_figure(fig, filename):
    """Save each diagnostic through the same output contract.
    
    Centralized saving gives the validation cell something concrete to check and
    keeps the notebook reproducible from a clean run."""
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=160, bbox_inches="tight")
    print("Saved:", path.name)


def to_uint8_mask(mask):
    """Translate Boolean logic into the mask format OpenCV expects.
    
    Inside Python, True/False is convenient. OpenCV morphology and component tools
    expect uint8 values 0 and 255.
    
    This helper makes that boundary explicit so mask semantics never change
    silently."""
    return (
        np.asarray(mask).astype(bool) * 255
    ).astype(np.uint8)


def overlay_mask(image_rgb, mask, alpha=0.35):
    """Show the segmentation without hiding the evidence underneath it.
    
    We blend red over foreground pixels instead of replacing them completely.
    With moderate alpha, the reader can inspect both the predicted region and the
    original image structure at the same time."""
    # Promote the overlay canvas to float so alpha blending does not clip intermediate values.
    output = image_rgb.astype(np.float32).copy()
    # Convert arbitrary mask encoding to one Boolean foreground convention before blending.
    mask_bool = np.asarray(mask).astype(bool)

    red = np.zeros_like(output)
    red[..., 0] = 255

    output[mask_bool] = (
        (1 - alpha) * output[mask_bool]
        + alpha * red[mask_bool]
    )

    return np.clip(output, 0, 255).astype(np.uint8)

## 1. Segmentation Problem Formulation

In [ ]:
def threshold_mask(
    image,
    threshold,
    foreground="dark",
):
    """Turn grayscale intensity into a Boolean foreground/background decision.

    Segmentation begins by asking one question for every pixel:

        "Does this pixel belong to the object?"

    For a dark target, pixels below the threshold are foreground.
    For a bright target, pixels at or above the threshold are foreground.
    """
    image = np.asarray(image)

    # Thresholding here is defined only for one scalar intensity per pixel.
    if image.ndim != 2:
        raise ValueError("threshold_mask expects a 2-D grayscale image.")

    if foreground == "dark":
        return image < threshold

    if foreground == "bright":
        return image >= threshold

    raise ValueError("foreground must be 'dark' or 'bright'.")


# Keep mask semantics explicit throughout the notebook.
FOREGROUND = True
BACKGROUND = False

print(
    "Mask convention:",
    {"foreground": FOREGROUND, "background": BACKGROUND},
)


## 2. Load the Lab Images


In [ ]:
# Load the role-specific inputs selected from the discovered dataset.
hand = load_gray(INPUT_IMAGES["hand"])
tower_rgb = load_rgb(INPUT_IMAGES["tower"])
peppers_rgb = load_rgb(INPUT_IMAGES["peppers"])

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Hand"
)

show_rgb(
    axes[1],
    tower_rgb,
    "Tower"
)

show_rgb(
    axes[2],
    peppers_rgb,
    "Peppers"
)

fig.tight_layout()
save_figure(
    fig,
    "01_input_images.png"
)
plt.show()

print("hand shape   :", hand.shape)
print("tower shape  :", tower_rgb.shape)
print("peppers shape:", peppers_rgb.shape)

## 3. Histogram-Based Threshold Selection


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4)
)

show_gray(
    axes[0],
    hand,
    "Hand image"
)

# Inspect the intensity distribution before selecting any thresholding strategy.
axes[1].hist(
    hand.ravel(),
    bins=256,
    range=(0, 255)
)
axes[1].set_title(
    "Hand intensity histogram"
)
axes[1].set_xlabel(
    "Intensity"
)
axes[1].set_ylabel(
    "Pixel count"
)

fig.tight_layout()
save_figure(
    fig,
    "02_hand_histogram.png"
)
plt.show()


## 4. Manual Global Thresholding


In [ ]:
# Establish a transparent manual baseline before automatic threshold selection.
# Use one fixed illustrative baseline; later Otsu removes this manual decision.
manual_threshold = 120

# The hand is darker than its background, so foreground corresponds to intensities below the threshold.
mask_manual = threshold_mask(hand, manual_threshold, foreground="dark")

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Original"
)

show_gray(
    axes[1],
    mask_manual,
    f"Mask — T={manual_threshold}"
)

# Keep original hand intensities only inside the mask so segmentation effect is visually inspectable.
masked_hand = np.where(
    mask_manual,
    hand,
    255
)

show_gray(
    axes[2],
    masked_hand,
    "Segmented foreground"
)

fig.tight_layout()
save_figure(
    fig,
    "03_manual_threshold.png"
)
plt.show()


## 5. Threshold Sensitivity


In [ ]:
# Sweep plausible thresholds to reveal sensitivity of the foreground mask.
# Sweep values on both sides of the baseline to expose threshold sensitivity.
thresholds = [
    70,
    100,
    130,
    160
]

fig, axes = plt.subplots(
    1,
    len(thresholds),
    figsize=(16, 4)
)

for ax, threshold in zip(
    axes,
    thresholds
):
    # Recompute the binary foreground at each threshold to expose sensitivity of area and boundaries.
    mask = hand < threshold

    show_gray(
        ax,
        mask,
        f"T={threshold}"
    )

fig.tight_layout()
save_figure(
    fig,
    "04_threshold_sensitivity.png"
)
plt.show()


## 6. Otsu Thresholding


In [ ]:
# Invert because the hand is darker than the background.
otsu_threshold, mask_otsu_cv = cv2.threshold(
    hand,
    0,
    255,
    cv2.THRESH_BINARY_INV
    + cv2.THRESH_OTSU
)

# Convert OpenCV's 0/255 threshold output to Boolean semantics used by later morphology.
mask_otsu = (
    mask_otsu_cv > 0
)

print(
    "Otsu threshold:",
    otsu_threshold
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Original"
)

show_gray(
    axes[1],
    mask_otsu,
    f"Otsu mask — T={otsu_threshold:.1f}"
)

overlay = overlay_mask(
    np.stack([hand] * 3, axis=-1),
    mask_otsu
)

show_rgb(
    axes[2],
    overlay,
    "Mask overlay"
)

fig.tight_layout()
save_figure(
    fig,
    "05_otsu_threshold.png"
)
plt.show()


## 7. Gaussian Smoothing Before Thresholding


In [ ]:
# Reduce local noise before Otsu to test whether class separation becomes more stable.
hand_blurred = cv2.GaussianBlur(
    # A 5x5 Gaussian removes local noise while preserving the hand boundary scale.
    hand,
    (5, 5),
    0
)

blur_otsu_threshold, blur_otsu_cv = cv2.threshold(
    hand_blurred,
    0,
    255,
    cv2.THRESH_BINARY_INV
    + cv2.THRESH_OTSU
)

blur_otsu = (
    blur_otsu_cv > 0
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Original"
)

show_gray(
    axes[1],
    hand_blurred,
    "Gaussian-smoothed"
)

show_gray(
    axes[2],
    blur_otsu,
    "Otsu after smoothing"
)

fig.tight_layout()
save_figure(
    fig,
    "06_smoothing_before_otsu.png"
)
plt.show()


## 8. Adaptive Thresholding


In [ ]:
# Compare local thresholds under the same neighborhood and offset.
# A 31x31 neighborhood captures local illumination; C=5 rejects marginal dark noise.
adaptive_mean = cv2.adaptiveThreshold(
    hand,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY_INV,
    31,
    5
)

adaptive_gaussian = cv2.adaptiveThreshold(
    hand,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV,
    31,
    5
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Original"
)

show_gray(
    axes[1],
    adaptive_mean,
    "Adaptive mean"
)

show_gray(
    axes[2],
    adaptive_gaussian,
    "Adaptive Gaussian"
)

fig.tight_layout()
save_figure(
    fig,
    "07_adaptive_thresholding.png"
)
plt.show()


## 9. Morphological Processing

In [ ]:
def apply_morphology(
    mask,
    operation,
    kernel,
    iterations=1,
):
    """Apply one binary morphology operation through a common interface.

    Morphology becomes easier to reason about when three decisions stay visible:

        - the input mask;
        - the structuring element;
        - the operation applied to that geometry.

    iterations controls how strongly the chosen operation is repeated.
    """
    mask_u8 = to_uint8_mask(mask)

    # Zero iterations would silently turn the requested operation into a no-op.
    if iterations < 1:
        raise ValueError("iterations must be at least 1.")

    if operation == "erode":
        result = cv2.erode(mask_u8, kernel, iterations=iterations)
    elif operation == "dilate":
        result = cv2.dilate(mask_u8, kernel, iterations=iterations)
    elif operation == "open":
        result = cv2.morphologyEx(
            mask_u8, cv2.MORPH_OPEN, kernel, iterations=iterations
        )
    elif operation == "close":
        result = cv2.morphologyEx(
            mask_u8, cv2.MORPH_CLOSE, kernel, iterations=iterations
        )
    else:
        raise ValueError(
            "operation must be 'erode', 'dilate', 'open', or 'close'."
        )

    return result > 0


print("Supported morphology:", ["erode", "dilate", "open", "close"])


## 10. Structuring Elements


In [ ]:
# Compare structuring-element geometry before applying morphology to the mask.
# Keep all candidate structuring elements at 7x7 so shape—not size—is compared.
# Keep all candidate elements at 7x7 so only geometry—not scale—changes.
kernel_rect = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (7, 7)
)

# Elliptical support follows rounded object boundaries with less directional bias than a rectangle.
kernel_ellipse = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (7, 7)
)

# Cross support is intentionally directional, making structuring-element geometry easy to compare.
kernel_cross = cv2.getStructuringElement(
    cv2.MORPH_CROSS,
    (7, 7)
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(10, 3)
)

show_gray(
    axes[0],
    kernel_rect,
    "Rectangle"
)

show_gray(
    axes[1],
    kernel_ellipse,
    "Ellipse"
)

show_gray(
    axes[2],
    kernel_cross,
    "Cross"
)

fig.tight_layout()
save_figure(
    fig,
    "08_structuring_elements.png"
)
plt.show()

## 11. Erosion and Dilation


In [ ]:
# Convert the Boolean mask once because OpenCV morphology expects 8-bit 0/255 input.
mask_uint8 = to_uint8_mask(
    blur_otsu
)

# Keep size fixed so the experiment isolates the morphological operator itself.
# Elliptical support reduces directional bias on rounded object boundaries.
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (7, 7)
)

eroded = cv2.erode(
    mask_uint8,
    kernel,
    iterations=1
)

dilated = cv2.dilate(
    mask_uint8,
    kernel,
    iterations=1
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    mask_uint8,
    "Original mask"
)

show_gray(
    axes[1],
    eroded,
    "Erosion"
)

show_gray(
    axes[2],
    dilated,
    "Dilation"
)

fig.tight_layout()
save_figure(
    fig,
    "09_erosion_dilation.png"
)
plt.show()


## 12. Opening and Closing


In [ ]:
# Contrast opening and closing on exactly the same binary input and kernel.
opened = cv2.morphologyEx(
    mask_uint8,
    cv2.MORPH_OPEN,
    kernel
)

closed = cv2.morphologyEx(
    mask_uint8,
    cv2.MORPH_CLOSE,
    kernel
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    mask_uint8,
    "Original mask"
)

show_gray(
    axes[1],
    opened,
    "Opening"
)

show_gray(
    axes[2],
    closed,
    "Closing"
)

fig.tight_layout()
save_figure(
    fig,
    "10_opening_closing.png"
)
plt.show()


## 13. Morphological Gradient


In [ ]:
# Extract a morphology-based boundary band without using image derivatives.
morph_gradient = cv2.morphologyEx(
    mask_uint8,
    cv2.MORPH_GRADIENT,
    kernel
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

show_gray(
    axes[0],
    mask_uint8,
    "Mask"
)

show_gray(
    axes[1],
    morph_gradient,
    "Morphological gradient"
)

fig.tight_layout()
save_figure(
    fig,
    "11_morphological_gradient.png"
)
plt.show()

## 14. Hole Filling


In [ ]:
# Fill only enclosed background regions without expanding the external boundary.
mask_with_holes = blur_otsu.astype(bool)

filled = ndimage.binary_fill_holes(
    mask_with_holes
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    mask_with_holes,
    "Before filling"
)

show_gray(
    axes[1],
    filled,
    "After filling"
)

show_gray(
    axes[2],
    filled.astype(int)
    - mask_with_holes.astype(int),
    "Pixels added"
)

fig.tight_layout()
save_figure(
    fig,
    "12_hole_filling.png"
)
plt.show()


## 15. Connected Components


In [ ]:
# Convert the cleaned mask into measurable object-level regions.
# Use 8-connectivity so diagonally touching foreground pixels form one region.
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    to_uint8_mask(filled),
    connectivity=8
)

print(
    "Number of foreground components:",
    num_labels - 1
)

# Exclude background label 0 and inspect only foreground component sizes.
component_areas = stats[
    1:,
    cv2.CC_STAT_AREA
]

print(
    "Foreground areas:",
    component_areas
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

show_gray(
    axes[0],
    filled,
    "Binary mask"
)

axes[1].imshow(
    labels,
    cmap="nipy_spectral"
)
axes[1].set_title(
    "Connected-component labels"
)
axes[1].axis("off")

fig.tight_layout()
save_figure(
    fig,
    "13_connected_components.png"
)
plt.show()


## 16. Remove Small Components


In [ ]:
# Express noise rejection as an explicit region-area criterion.
# 500 px removes tiny artifacts while remaining far below the target hand area.
minimum_area = 500

# Begin with an empty Boolean mask so only components meeting the area prior are copied in.
clean_components = np.zeros_like(
    labels,
    dtype=bool
)

for label_id in range(
    1,
    num_labels
):
    area = stats[
        label_id,
        cv2.CC_STAT_AREA
    ]

    # Keep only components that satisfy the explicit object-size prior.
    if area >= minimum_area:
        clean_components |= (
            labels == label_id
        )

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

show_gray(
    axes[0],
    filled,
    "Before area filtering"
)

show_gray(
    axes[1],
    clean_components,
    f"Area ≥ {minimum_area}"
)

fig.tight_layout()
save_figure(
    fig,
    "14_component_area_filtering.png"
)
plt.show()


## 17. Contours


In [ ]:
# Convert cleaned connected regions into explicit boundary representations.
contours, hierarchy = cv2.findContours(
    to_uint8_mask(
        clean_components
    ),
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

hand_rgb = np.stack(
    [hand] * 3,
    axis=-1
)

# Draw contours on a copy so visualization never alters the original RGB evidence.
contour_view = hand_rgb.copy()

cv2.drawContours(
    contour_view,
    contours,
    -1,
    (255, 0, 0),
    2
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

show_gray(
    axes[0],
    clean_components,
    "Clean mask"
)

show_rgb(
    axes[1],
    contour_view,
    "Detected contours"
)

fig.tight_layout()
save_figure(
    fig,
    "15_contours.png"
)
plt.show()

print(
    "Number of external contours:",
    len(contours)
)

## 18. Region Properties


In [ ]:
# Measure geometry per contour so region filtering can use object-level evidence.
region_rows = []

for index, contour in enumerate(
    contours,
    start=1
):
    # Region area is the primary size descriptor used to judge whether a contour is meaningful.
    area = cv2.contourArea(
        contour
    )

    perimeter = cv2.arcLength(
        contour,
        True
    )

    x, y, w, h = cv2.boundingRect(
        contour
    )

    moments = cv2.moments(
        contour
    )

    # Centroid division is valid only for contours with non-zero spatial mass.
    if moments["m00"] != 0:
        cx = moments["m10"] / moments["m00"]
        cy = moments["m01"] / moments["m00"]
    else:
        cx = np.nan
        cy = np.nan

    circularity = (
        4 * np.pi * area
        / (perimeter ** 2)
        # Compactness is undefined for zero-perimeter degenerate contours.
        if perimeter > 0
        else np.nan
    )

    # Aspect ratio summarizes component geometry independently of its absolute pixel size.
    aspect_ratio = (
        w / h
        # Aspect ratio requires non-zero bounding-box height.
        if h > 0
        else np.nan
    )

    region_rows.append(
        {
            "component": index,
            "area": area,
            "perimeter": perimeter,
            "cx": cx,
            "cy": cy,
            "width": w,
            "height": h,
            "aspect_ratio": aspect_ratio,
            "circularity": circularity,
        }
    )

for row in region_rows:
    print(row)

## 19. Color Segmentation


In [ ]:
# Separate hue from brightness before defining chromatic segmentation rules.
peppers_hsv = cv2.cvtColor(
    peppers_rgb,
    cv2.COLOR_RGB2HSV
)

hue = peppers_hsv[..., 0]
# Inspect saturation separately because weakly saturated pixels should not be treated as reliable color evidence.
saturation = peppers_hsv[..., 1]
value = peppers_hsv[..., 2]

fig, axes = plt.subplots(
    1,
    4,
    figsize=(16, 4)
)

show_rgb(
    axes[0],
    peppers_rgb,
    "RGB"
)

show_gray(
    axes[1],
    hue,
    "Hue"
)

show_gray(
    axes[2],
    saturation,
    "Saturation"
)

show_gray(
    axes[3],
    value,
    "Value"
)

fig.tight_layout()
save_figure(
    fig,
    "16_hsv_channels.png"
)
plt.show()

### HSV Range Experiment


In [ ]:
# Red wraps around the HSV hue axis, so two intervals are required.
# Saturation/value floors reject pale or very dark pixels that are not reliably red.
# Hue bounds target red near both ends of OpenCV's circular 0–179 hue range.
lower_red_1 = np.array(
    [0, 80, 50],
    dtype=np.uint8
)

upper_red_1 = np.array(
    [12, 255, 255],
    dtype=np.uint8
)

lower_red_2 = np.array(
    [165, 80, 50],
    dtype=np.uint8
)

upper_red_2 = np.array(
    [179, 255, 255],
    dtype=np.uint8
)

# First hue interval captures red near the low end of OpenCV's circular hue axis.
mask_red_1 = cv2.inRange(
    peppers_hsv,
    lower_red_1,
    upper_red_1
)

# Second interval captures red that wraps around near hue 179.
mask_red_2 = cv2.inRange(
    peppers_hsv,
    lower_red_2,
    upper_red_2
)

# Merge both hue intervals because they represent the same physical color class.
red_mask = (
    (mask_red_1 > 0)
    | (mask_red_2 > 0)
)

red_segment = peppers_rgb.copy()
red_segment[~red_mask] = 0

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_rgb(
    axes[0],
    peppers_rgb,
    "Original peppers"
)

show_gray(
    axes[1],
    red_mask,
    "Red-color mask"
)

show_rgb(
    axes[2],
    red_segment,
    "Segmented red regions"
)

fig.tight_layout()
save_figure(
    fig,
    "17_color_segmentation.png"
)
plt.show()


## 20. Edge-Based Segmentation


In [ ]:
tower_gray = cv2.cvtColor(
    tower_rgb,
    cv2.COLOR_RGB2GRAY
)

# Pre-smooth the tower so Canny responds to structural edges rather than pixel noise.
tower_blur = cv2.GaussianBlur(
    tower_gray,
    (5, 5),
    0
)

# Use a 1:2 hysteresis ratio to keep strong edges and supported weak edges.
tower_edges = cv2.Canny(
    tower_blur,
    80,
    160
)

# A 5x5 closing kernel bridges short gaps without merging distant structures.
edge_kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (5, 5)
)

# Two closing passes connect fragmented edge segments while limiting thickening.
tower_edges_closed = cv2.morphologyEx(
    tower_edges,
    cv2.MORPH_CLOSE,
    edge_kernel,
    iterations=2
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_rgb(
    axes[0],
    tower_rgb,
    "Tower"
)

show_gray(
    axes[1],
    tower_edges,
    "Canny edges"
)

show_gray(
    axes[2],
    tower_edges_closed,
    "Closed edge map"
)

fig.tight_layout()
save_figure(
    fig,
    "18_edge_based_segmentation.png"
)
plt.show()


## 21. Distance Transform


In [ ]:
# Interior distance peaks provide markers for separating touching regions.
# A 5x5 Euclidean mask gives smoother distance estimates than the 3x3 option.
distance = cv2.distanceTransform(
    to_uint8_mask(clean_components),
    cv2.DIST_L2,
    5
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

show_gray(
    axes[0],
    clean_components,
    "Binary mask"
)

im = axes[1].imshow(
    distance,
    cmap="viridis"
)

axes[1].set_title(
    "Distance transform"
)

axes[1].axis("off")

fig.colorbar(
    im,
    ax=axes[1],
    fraction=0.046
)

fig.tight_layout()
save_figure(
    fig,
    "19_distance_transform.png"
)
plt.show()


## 22. Watershed Segmentation


In [ ]:
# Preserve a clean RGB copy because watershed writes boundaries only into the visualization stage.
watershed_input = peppers_rgb.copy()

peppers_gray = cv2.cvtColor(
    peppers_rgb,
    cv2.COLOR_RGB2GRAY
)

_, peppers_binary = cv2.threshold(
    peppers_gray,
    0,
    255,
    cv2.THRESH_BINARY
    + cv2.THRESH_OTSU
)

# A 3x3 kernel keeps marker cleanup local and limits geometric drift.
ws_kernel = np.ones(
    (3, 3),
    np.uint8
)

opening = cv2.morphologyEx(
    peppers_binary,
    cv2.MORPH_OPEN,
    ws_kernel,
    iterations=2
)

# Extra dilation builds a conservative region known to be background.
sure_bg = cv2.dilate(
    opening,
    ws_kernel,
    # Three background dilations deliberately create a conservative sure-background region.
iterations=3
)

# Distance-to-background values identify deep interior pixels suitable for confident object markers.
dist_transform = cv2.distanceTransform(
    opening,
    cv2.DIST_L2,
    5
)

# Keep only deep interior pixels as high-confidence foreground markers.
_, sure_fg = cv2.threshold(
    dist_transform,
    0.5 * dist_transform.max(),
    255,
    0
)

sure_fg = np.uint8(
    sure_fg
)

# Pixels that are neither sure foreground nor sure background remain unknown.
unknown = cv2.subtract(
    sure_bg,
    sure_fg
)

num_markers, markers = cv2.connectedComponents(
    sure_fg
)

# Reserve label 0 for unknown pixels, as required by OpenCV watershed.
markers = markers + 1
markers[unknown == 255] = 0

markers_ws = cv2.watershed(
    cv2.cvtColor(
        watershed_input,
        cv2.COLOR_RGB2BGR
    ),
    markers.copy()
)

# Keep watershed labels separate from the RGB visualization so boundaries can be overlaid safely.
watershed_view = watershed_input.copy()
watershed_view[
    markers_ws == -1
] = [255, 0, 0]

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

show_rgb(
    axes[0, 0],
    peppers_rgb,
    "Input"
)

show_gray(
    axes[0, 1],
    opening,
    "Opening"
)

show_gray(
    axes[0, 2],
    sure_bg,
    "Sure background"
)

show_gray(
    axes[1, 0],
    dist_transform,
    "Distance transform"
)

show_gray(
    axes[1, 1],
    sure_fg,
    "Sure foreground"
)

show_rgb(
    axes[1, 2],
    watershed_view,
    "Watershed boundaries"
)

fig.tight_layout()
save_figure(
    fig,
    "20_watershed.png"
)
plt.show()


## 23. Ground Truth and Segmentation Metrics


In [ ]:
def segmentation_metrics(
    ground_truth,
    prediction
):
    """Ask several different questions about one predicted mask.

    Accuracy counts every pixel, but background can dominate and make it look good
    too easily. Precision asks how much predicted foreground is correct. Recall
    asks how much true foreground was recovered. IoU and Dice measure overlap
    directly.

    We compute all of them because no single number describes every segmentation
    failure mode.
    """
    gt = np.asarray(ground_truth).astype(bool)
    pred = np.asarray(prediction).astype(bool)

    # Confusion counts come directly from Boolean set relationships.
    tp = np.logical_and(gt, pred).sum()
    tn = np.logical_and(~gt, ~pred).sum()
    fp = np.logical_and(~gt, pred).sum()
    fn = np.logical_and(gt, ~pred).sum()

    # Epsilon only protects empty-mask denominators; it is negligible otherwise.
    epsilon = 1e-12

    # Accuracy is retained for completeness but interpreted with foreground metrics.
    accuracy = (tp + tn) / (tp + tn + fp + fn + epsilon)
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)
    iou = tp / (tp + fp + fn + epsilon)
    dice = 2 * tp / (2 * tp + fp + fn + epsilon)

    return {
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "Accuracy": float(accuracy),
        "Precision": float(precision),
        "Recall": float(recall),
        "IoU": float(iou),
        "Dice": float(dice),
    }


### Synthetic Ground-Truth Check


In [ ]:
# No official hand.png ground truth is provided; this synthetic case validates metric behavior only.
# Use a controlled perturbed mask to verify metric behavior against known ground truth.
demo_gt = clean_components.copy()

demo_prediction = cv2.erode(
    to_uint8_mask(
        clean_components
    ),
    np.ones(
        (7, 7),
        np.uint8
    ),
    iterations=1
) > 0

demo_metrics = segmentation_metrics(
    demo_gt,
    demo_prediction
)

for key, value in demo_metrics.items():
    # Format continuous metrics separately from integer confusion counts.
    if isinstance(value, float):
        print(
            f"{key:10s}: {value:.4f}"
        )
    else:
        print(
            f"{key:10s}: {value}"
        )


## 24. Dice vs IoU Relationship

In [ ]:
# Dice and IoU use different scales to describe the same binary overlap.
demo_iou = demo_metrics["IoU"]
demo_dice = demo_metrics["Dice"]

# Convert each metric analytically into the other.
dice_from_iou = 2.0 * demo_iou / (1.0 + demo_iou)
iou_from_dice = demo_dice / (2.0 - demo_dice)

assert np.isclose(dice_from_iou, demo_dice)
assert np.isclose(iou_from_dice, demo_iou)

print(f"IoU           : {demo_iou:.4f}")
print(f"Dice          : {demo_dice:.4f}")
print(f"Dice from IoU : {dice_from_iou:.4f}")
print(f"IoU from Dice : {iou_from_dice:.4f}")


## 25. Under-Segmentation vs Over-Segmentation

In [ ]:
# Use one controlled kernel so only the failure direction changes.
comparison_kernel = np.ones((9, 9), dtype=np.uint8)

# Erosion removes valid foreground and simulates under-segmentation.
under_segmented = cv2.erode(
    to_uint8_mask(demo_gt),
    comparison_kernel,
    iterations=1,
) > 0

# Dilation adds background around the object and simulates over-segmentation.
over_segmented = cv2.dilate(
    to_uint8_mask(demo_gt),
    comparison_kernel,
    iterations=1,
) > 0

under_metrics = segmentation_metrics(demo_gt, under_segmented)
over_metrics = segmentation_metrics(demo_gt, over_segmented)

print(
    "Under-segmentation:",
    f"Precision={under_metrics['Precision']:.4f}",
    f"Recall={under_metrics['Recall']:.4f}",
    f"IoU={under_metrics['IoU']:.4f}",
)

print(
    "Over-segmentation:",
    f"Precision={over_metrics['Precision']:.4f}",
    f"Recall={over_metrics['Recall']:.4f}",
    f"IoU={over_metrics['IoU']:.4f}",
)


## 26. End-to-End Binary Segmentation Pipeline


In [ ]:
# Compose the validated stages into one reusable segmentation pipeline.
def segment_dark_object(
    image_gray,
    blur_kernel=(5, 5),
    minimum_area=500
):
    """Turn a dark object into a clean, measurable binary region step by step.

    The pipeline deliberately stays transparent:

        - Blur small intensity noise.
        - Use inverted Otsu because the target is darker than its background.
        - Open/close the mask to remove specks and repair small gaps.
        - Fill internal holes.
        - Label connected regions.
        - Remove components that are too small to be the target.

    The 500-pixel area rule belongs to this image scale. On a different
    resolution, that prior must be reconsidered rather than copied blindly.
    """
    # A 5x5 default window stabilizes Otsu without erasing the main hand boundary.
    blurred = cv2.GaussianBlur(image_gray, blur_kernel, 0)

    threshold_value, binary = cv2.threshold(
        blurred,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
    )

    # A compact ellipse regularizes boundaries with limited directional bias.
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5, 5),
    )

    cleaned = apply_morphology(
        binary > 0,
        operation="open",
        kernel=kernel,
        iterations=1,
    )

    cleaned = apply_morphology(
        cleaned,
        operation="close",
        kernel=kernel,
        iterations=1,
    )

    filled = ndimage.binary_fill_holes(cleaned)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        to_uint8_mask(filled),
        connectivity=8,
    )

    # Start empty so only components satisfying the explicit size prior are retained.
    final_mask = np.zeros_like(filled, dtype=bool)

    for label_id in range(1, num_labels):
        area = stats[label_id, cv2.CC_STAT_AREA]

        # Keep only regions large enough to be plausible target components.
        if area >= minimum_area:
            final_mask |= labels == label_id

    return {
        "threshold": float(threshold_value),
        "blurred": blurred,
        "binary": binary > 0,
        "cleaned": cleaned,
        "filled": filled,
        "mask": final_mask,
    }


## 27. Segmentation Method Selection Criteria

In [ ]:
# Make method selection explicit: choose the algorithm family from the dominant image difficulty.
def recommend_segmentation_method(
    uneven_illumination=False,
    color_is_discriminative=False,
    touching_objects=False,
):
    """Choose a classical method from the dominant evidence in the image.

    This is a decision aid, not an optimizer. Its purpose is to make the
    assumptions behind method selection explicit rather than habitual.
    """
    # Touching regions need object-separation geometry, not only a pixel threshold.
    if touching_objects:
        return "distance transform + marker-controlled watershed"

    # Strong chromatic separation should be exploited before discarding color information.
    if color_is_discriminative:
        return "color-space thresholding (for example HSV)"

    # Local illumination changes violate the single-threshold assumption.
    if uneven_illumination:
        return "adaptive thresholding"

    return "global thresholding / Otsu"


# Exercise the decision rules on representative segmentation scenarios.
selection_examples = {
    "uniform grayscale object": recommend_segmentation_method(),
    "uneven lighting": recommend_segmentation_method(
        uneven_illumination=True
    ),
    "distinctive color": recommend_segmentation_method(
        color_is_discriminative=True
    ),
    "touching objects": recommend_segmentation_method(
        touching_objects=True
    ),
}

for case, method in selection_examples.items():
    print(f"{case:24s} -> {method}")


## 28. Integrated Segmentation Workflow

In [ ]:
# Run the complete pipeline once and expose every intermediate stage for inspection.
# Reuse the same 500-pixel area prior from the earlier component-analysis experiment.
hand_result = segment_dark_object(
    hand,
    minimum_area=500
)

# Extract the pipeline's final mask separately so it can be visualized and validated against intermediates.
final_hand_mask = hand_result[
    "mask"
]

final_overlay = overlay_mask(
    np.stack(
        [hand] * 3,
        axis=-1
    ),
    final_hand_mask
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

show_gray(
    axes[0, 0],
    hand,
    "Input"
)

show_gray(
    axes[0, 1],
    hand_result["blurred"],
    "Smoothed"
)

show_gray(
    axes[0, 2],
    hand_result["binary"],
    "Otsu threshold"
)

show_gray(
    axes[1, 0],
    hand_result["cleaned"],
    "Morphological cleanup"
)

show_gray(
    axes[1, 1],
    final_hand_mask,
    "Final mask"
)

show_rgb(
    axes[1, 2],
    final_overlay,
    "Final overlay"
)

fig.tight_layout()
save_figure(
    fig,
    "21_complete_pipeline.png"
)
plt.show()

print(
    "Pipeline Otsu threshold:",
    hand_result["threshold"]
)


## Final Analysis & Interpretation

### Main findings

- Histogram structure determines whether one global intensity threshold is plausible.
- Manual, Otsu, and adaptive thresholding represent different assumptions about class separation and illumination.
- Morphology changes mask geometry according to structuring-element size and shape.
- Connected components convert pixel masks into measurable regions that can be filtered by object-level properties.
- HSV segmentation exploits chromatic information that grayscale cannot represent.
- Edge maps provide boundary evidence but generally need additional processing before becoming usable region masks.
- Distance transforms and watershed support separation of touching objects through marker geometry.
- Precision, recall, IoU, and Dice are necessary complements to raw pixel accuracy.

### Engineering interpretation

Segmentation quality depends on the assumptions made before the final mask appears: representation, threshold strategy, morphology, connectivity, region priors, and evaluation metric all matter.

### Limitations

Several thresholds and priors are tuned to the supplied examples, including the area criterion and HSV bounds. They should be re-estimated when resolution, scale, illumination, or acquisition conditions change.

### Final conclusion

The notebook provides a transparent classical segmentation pipeline from thresholding through morphology, regions, color, edges, watershed, metrics, failure-mode analysis, and final validation.
